## Example 1: Basic Command-Line Parsing with docopt

In [ ]:
from docopt import docopt, DocoptExit
import docopt as doc_module
import sys

print("Example 1: Basic Command-Line Parsing with docopt")

try:
    print("✓ Basic command-line parsing:")
    
    # 1. Package information and basic usage
    print("  1. Package information:")
    
    print(f"    docopt package: {doc_module.__name__}")
    print(f"    docopt version: {getattr(doc_module, '__version__', 'unknown')}")
    
    # Show available functions
    public_attrs = [attr for attr in dir(doc_module) if not attr.startswith('_')]
    print(f"    Available functions: {public_attrs}")
    
    # 2. Simple command-line interface definition
    print("  2. Simple CLI definition:")
    
    # Define a simple docstring for command-line interface
    simple_doc = """Simple Calculator
    
Usage:
  calculator add <x> <y>
  calculator subtract <x> <y>
  calculator multiply <x> <y>
  calculator divide <x> <y>
  calculator --version
  calculator (-h | --help)

Options:
  -h --help     Show this screen.
  --version     Show version.
"""
    
    print("    Docstring example:")
    print("      Usage patterns:")
    usage_lines = [line.strip() for line in simple_doc.split('\n') if line.strip().startswith('calculator')]
    for line in usage_lines[:4]:  # Show first 4 usage patterns
        print(f"        {line}")
    
    print("    Options defined:")
    option_lines = [line.strip() for line in simple_doc.split('\n') if line.strip().startswith(('-h', '--'))]
    for line in option_lines:
        print(f"        {line}")
    
    # 3. Parse different command combinations
    print("  3. Command parsing examples:")
    
    # Test different command-line combinations
    test_commands = [
        ['add', '5', '3'],
        ['subtract', '10', '4'],
        ['multiply', '2', '7'],
        ['divide', '20', '4'],
        ['--version'],
        ['--help']
    ]
    
    for i, args in enumerate(test_commands):
        try:
            print(f"    Test {i+1}: {' '.join(args)}")
            
            # Parse the arguments using docopt
            # Note: In Jupyter, we simulate sys.argv
            parsed_args = docopt(simple_doc, argv=args, version='Calculator 1.0')
            
            print(f"      Parsed arguments:")
            for key, value in parsed_args.items():
                if value:  # Only show non-None/non-False values
                    print(f"        {key}: {value}")
            
            # Simulate command execution based on parsed args
            if parsed_args['add']:
                x, y = float(parsed_args['<x>']), float(parsed_args['<y>'])
                result = x + y
                print(f"      Result: {x} + {y} = {result}")
            elif parsed_args['subtract']:
                x, y = float(parsed_args['<x>']), float(parsed_args['<y>'])
                result = x - y
                print(f"      Result: {x} - {y} = {result}")
            elif parsed_args['multiply']:
                x, y = float(parsed_args['<x>']), float(parsed_args['<y>'])
                result = x * y
                print(f"      Result: {x} × {y} = {result}")
            elif parsed_args['divide']:
                x, y = float(parsed_args['<x>']), float(parsed_args['<y>'])
                if y != 0:
                    result = x / y
                    print(f"      Result: {x} ÷ {y} = {result}")
                else:
                    print(f"      Error: Division by zero")
            elif parsed_args['--version']:
                print(f"      Version information displayed")
            elif parsed_args['--help']:
                print(f"      Help information would be displayed")
            
        except SystemExit:
            # docopt calls sys.exit() for --help and --version
            print(f"      System exit triggered (help/version)")
        except Exception as e:
            print(f"      Parsing failed: {type(e).__name__}: {e}")
            raise
        
        print()
    
    # 4. Argument validation and types
    print("  4. Argument validation:")
    
    # Test invalid arguments: an argv that matches no usage pattern must be rejected with DocoptExit
    invalid_tests = [
        (['add', '5'], 'Missing second number'),
        (['invalid', '1', '2'], 'Invalid command'),
    ]
    
    for args, description in invalid_tests:
        print(f"    Invalid test: {' '.join(args)} ({description})")
        try:
            docopt(simple_doc, argv=args)
        except DocoptExit:
            print(f"      Correctly rejected with DocoptExit")
        else:
            raise AssertionError(f"docopt accepted argv {args!r} ({description})")
    
    # docopt matches argv against the usage patterns only and does no type checking (measured: docopt 0.6.2
    # parses `add abc 5` with <x> == 'abc'), so validating the number is the application's job
    parsed_args = docopt(simple_doc, argv=['add', 'abc', '5'])
    assert parsed_args['add'] and parsed_args['<x>'] == 'abc', parsed_args
    print("    Number format: docopt passes <x>='abc' through unchanged; the application must validate it")
    
    # 5. Command structure analysis
    print("  5. Command structure analysis:")
    
    # Analyze the structure of parsed commands
    sample_parse = docopt(simple_doc, argv=['add', '1', '2'])
    
    print(f"    Sample parsing result structure:")
    print(f"      Total keys: {len(sample_parse)}")
    
    # Categorize the keys
    commands = [k for k, v in sample_parse.items() if k.islower() and not k.startswith('<') and not k.startswith('-')]
    arguments = [k for k in sample_parse.keys() if k.startswith('<') and k.endswith('>')]
    options = [k for k in sample_parse.keys() if k.startswith('-')]
    
    print(f"      Commands: {commands}")
    print(f"      Arguments: {arguments}")
    print(f"      Options: {options}")
    
    print(f"    Key-value pairs:")
    for key, value in sorted(sample_parse.items()):
        print(f"      '{key}': {value} ({type(value).__name__})")
    
    print("✓ Basic command-line parsing completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"docopt example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced Options and Pattern Matching

In [ ]:
from docopt import docopt, DocoptExit

print("Example 2: Advanced Options and Pattern Matching")

try:
    print("✓ Advanced options and pattern matching:")
    
    # 1. Complex CLI with multiple option types
    print("  1. Complex CLI definition:")
    
    # Define a more complex command-line interface
    advanced_doc = """File Processing Tool

Usage:
  fileprocess scan <directory> [--recursive] [--include=<patterns>...] [--exclude=<patterns>...]
  fileprocess convert <input> <output> [--format=<fmt>] [--quality=<level>] [--verbose]
  fileprocess backup <source> <destination> [--compress] [--encryption=<type>] [--threads=<n>]
  fileprocess status [--json] [--detailed]
  fileprocess config set <key> <value>
  fileprocess config get <key>
  fileprocess config list [--all]
  fileprocess (-h | --help)
  fileprocess --version

Options:
  -h --help                Show this help screen.
  --version                Show version.
  -r --recursive           Scan directories recursively.
  --include=<patterns>     Include file patterns (can be used multiple times).
  --exclude=<patterns>     Exclude file patterns (can be used multiple times).
  --format=<fmt>           Output format [default: auto].
  --quality=<level>        Quality level 1-10 [default: 8].
  -v --verbose             Enable verbose output.
  --compress               Enable compression for backup.
  --encryption=<type>      Encryption type (aes256, rsa) [default: none].
  --threads=<n>            Number of threads [default: 4].
  --json                   Output in JSON format.
  --detailed               Show detailed information.
  --all                    Show all configuration options.

Examples:
  fileprocess scan /home/user --recursive --include="*.py" --include="*.txt"
  fileprocess convert input.jpg output.png --format=png --quality=9
  fileprocess backup /data /backup --compress --threads=8
"""
    
    print("    Command categories:")
    commands = ['scan', 'convert', 'backup', 'status', 'config']
    for cmd in commands:
        print(f"      {cmd}: File {cmd} operations")
    
    # 2. Test different option combinations
    print("  2. Option combination testing:")
    
    test_cases = [
        # Scan command variations
        (['scan', '/home/user'], 'Basic scan'),
        (['scan', '/home/user', '--recursive'], 'Recursive scan'),
        (['scan', '/data', '--include=*.py', '--include=*.txt', '--exclude=*.pyc'], 'Pattern filtering'),
        
        # Convert command variations
        (['convert', 'image.jpg', 'output.png'], 'Basic conversion'),
        (['convert', 'photo.raw', 'photo.jpg', '--format=jpeg', '--quality=9', '--verbose'], 'Advanced conversion'),
        
        # Backup command variations
        (['backup', '/source', '/dest'], 'Basic backup'),
        (['backup', '/data', '/backup', '--compress', '--threads=16', '--encryption=aes256'], 'Advanced backup'),
        
        # Status and config commands
        (['status'], 'Basic status'),
        (['status', '--json', '--detailed'], 'Detailed JSON status'),
        (['config', 'set', 'max_threads', '8'], 'Set configuration'),
        (['config', 'get', 'format'], 'Get configuration'),
        (['config', 'list', '--all'], 'List all config'),
    ]
    
    for i, (args, description) in enumerate(test_cases):
        try:
            print(f"    Test {i+1}: {description}")
            print(f"      Command: {' '.join(args)}")
            
            parsed = docopt(advanced_doc, argv=args)
            
            # Show key parsed results
            active_items = [(k, v) for k, v in parsed.items() if v and v != 'none']
            print(f"      Active options: {len(active_items)}")
            
            # Group results by type
            commands_found = [k for k, v in active_items if k.islower() and not k.startswith('<') and not k.startswith('-')]
            arguments_found = [k for k, v in active_items if k.startswith('<')]
            options_found = [k for k, v in active_items if k.startswith('-')]
            
            if commands_found:
                print(f"        Commands: {commands_found}")
            if arguments_found:
                print(f"        Arguments: {[f'{k}={parsed[k]}' for k in arguments_found]}")
            if options_found:
                print(f"        Options: {[f'{k}={parsed[k]}' for k in options_found if parsed[k] != True]}")
                bool_options = [k for k in options_found if parsed[k] == True]
                if bool_options:
                    print(f"        Flags: {bool_options}")
            
        except SystemExit:
            print(f"      System exit (help/version/error)")
        except Exception as e:
            print(f"      Parse error: {type(e).__name__}")
            raise
        
        print()
    
    # 3. Multi-value options handling
    print("  3. Multi-value options:")
    
    # Test commands with repeated options
    multi_value_tests = [
        ['scan', '/data', '--include=*.py', '--include=*.js', '--include=*.html'],
        ['scan', '/src', '--exclude=*.pyc', '--exclude=*.tmp', '--exclude=__pycache__'],
    ]
    
    for args in multi_value_tests:
        try:
            print(f"    Command: {' '.join(args)}")
            parsed = docopt(advanced_doc, argv=args)
            
            # Check multi-value options
            include_patterns = parsed.get('--include', [])
            exclude_patterns = parsed.get('--exclude', [])
            
            if include_patterns:
                print(f"      Include patterns: {include_patterns}")
                print(f"      Pattern count: {len(include_patterns)}")
            if exclude_patterns:
                print(f"      Exclude patterns: {exclude_patterns}")
                print(f"      Pattern count: {len(exclude_patterns)}")
            
        except Exception as e:
            print(f"      Multi-value parse error: {type(e).__name__}")
            raise
        
        print()
    
    # 4. Default value handling
    print("  4. Default value handling:")
    
    # Test commands to show default values
    default_tests = [
        (['convert', 'in.jpg', 'out.png'], 'format and quality defaults'),
        (['backup', '/src', '/dst'], 'threads and encryption defaults'),
    ]
    
    for args, description in default_tests:
        try:
            print(f"    Test: {description}")
            print(f"      Command: {' '.join(args)}")
            
            parsed = docopt(advanced_doc, argv=args)
            
            # Show default values that were applied
            default_keys = ['--format', '--quality', '--threads', '--encryption']
            defaults_applied = []
            
            for key in default_keys:
                if key in parsed and parsed[key] and parsed[key] != 'none':
                    defaults_applied.append(f"{key}={parsed[key]}")
            
            if defaults_applied:
                print(f"      Defaults applied: {defaults_applied}")
            else:
                print(f"      No defaults applied")
                
        except Exception as e:
            print(f"      Default test error: {type(e).__name__}")
            raise
        
        print()
    
    # 5. Error handling and validation
    print("  5. Error handling:")
    
    # Test invalid command combinations
    error_tests = [
        (['scan'], 'Missing required directory argument'),
        (['convert', 'input.jpg'], 'Missing output file'),
        (['config', 'set', 'key'], 'Missing value for set'),
        (['invalid-command'], 'Invalid command'),
        (['scan', '/data', '--quality=5'], 'Invalid option for command'),
    ]
    
    for args, expected_error in error_tests:
        print(f"    Error test: {expected_error}")
        print(f"      Command: {' '.join(args)}")
        try:
            docopt(advanced_doc, argv=args)
        except DocoptExit:
            print(f"      Correctly handled: DocoptExit (docopt validation)")
        else:
            raise AssertionError(f"docopt accepted argv {args!r}; expected a rejection ({expected_error})")
        
        print()
    
    # 6. Pattern matching analysis
    print("  6. Pattern matching summary:")
    
    # Analyze the patterns used in the docstring
    usage_section = advanced_doc.split('Usage:')[1].split('Options:')[0]
    usage_lines = [line.strip() for line in usage_section.split('\n') if line.strip() and not line.strip().startswith('fileprocess (-h')]
    
    print(f"    Total usage patterns: {len(usage_lines)}")
    
    pattern_types = {
        'required_args': 0,
        'optional_args': 0,
        'flags': 0,
        'valued_options': 0,
        'repeated_options': 0
    }
    
    for line in usage_lines:
        if '<' in line and '>' in line:
            pattern_types['required_args'] += line.count('<')
        if '[' in line and ']' in line:
            pattern_types['optional_args'] += line.count('[')
        if '--' in line:
            pattern_types['flags'] += line.count('--')
        if '=' in line:
            pattern_types['valued_options'] += line.count('=')
        if '...' in line:
            pattern_types['repeated_options'] += line.count('...')
    
    print(f"    Pattern analysis:")
    for pattern_type, count in pattern_types.items():
        print(f"      {pattern_type.replace('_', ' ').title()}: {count}")
    
    print("✓ Advanced options and pattern matching completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"docopt example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Subcommands and Complex Command Structures

In [ ]:
from docopt import docopt

print("Example 3: Subcommands and Complex Command Structures")

try:
    print("✓ Subcommands and complex structures:")
    
    # 1. Git-like subcommand structure
    print("  1. Git-like CLI structure:")
    
    # Define a Git-like command interface with nested subcommands.
    # `status` takes --show-branch, not --branch: --branch is declared below as --branch=<name>
    # (it takes a value), so a bare `[--branch]` in a usage line makes docopt 0.6.2 consume the
    # closing `]` as that value and reject the whole doc with "unmatched '['".
    git_like_doc = """Git-like Version Control System

Usage:
  vcs init [<directory>] [--bare] [--template=<template>]
  vcs clone <repository> [<directory>] [--depth=<depth>] [--branch=<name>]
  vcs add <files>... [--all] [--force] [--dry-run]
  vcs commit [<files>...] (-m <message> | --message=<message>) [--all] [--amend]
  vcs branch [<name>] [--delete <branch>] [--rename <old> <new>] [--list]
  vcs checkout <branch> [--create] [--force]
  vcs merge <branch> [--no-ff] [--squash] [--abort]
  vcs push [<remote>] [<branch>] [--force] [--tags] [--dry-run]
  vcs pull [<remote>] [<branch>] [--rebase] [--no-commit]
  vcs status [--short] [--show-branch] [--porcelain]
  vcs log [<path>] [--oneline] [--graph] [--since=<date>] [--until=<date>] [-n <count>]
  vcs remote add <name> <url>
  vcs remote remove <name>
  vcs remote list [--verbose]
  vcs config get <key>
  vcs config set <key> <value> [--global] [--local]
  vcs config unset <key>
  vcs stash [push | pop | list | drop] [--include-untracked]
  vcs (-h | --help)
  vcs --version

Options:
  -h --help              Show this help message.
  --version              Show version.
  --bare                 Create a bare repository.
  --template=<template>  Use custom template directory.
  --depth=<depth>        Create shallow clone with history depth.
  --branch=<name>        Clone specific branch [default: main].
  --all                  Include all files.
  --force                Force operation.
  --dry-run              Show what would be done without executing.
  -m --message=<msg>     Commit message.
  --amend                Amend the previous commit.
  --create               Create new branch if it doesn't exist.
  --delete <branch>      Delete specified branch.
  --rename <old> <new>   Rename branch from <old> to <new>.
  --list                 List all branches.
  --no-ff                Force merge commit even if fast-forward is possible.
  --squash               Squash all commits from branch into single commit.
  --abort                Abort current merge.
  --tags                 Push tags along with commits.
  --rebase               Rebase instead of merge.
  --no-commit            Don't automatically commit after pull.
  --short                Show short status format.
  --show-branch          Show branch information in the status.
  --porcelain            Show porcelain (script-friendly) status format.
  --oneline              Show one commit per line.
  --graph                Show commit graph.
  --since=<date>         Show commits since date.
  --until=<date>         Show commits until date.
  -n <count>             Limit number of commits [default: 10].
  --verbose              Show verbose output.
  --global               Apply configuration globally.
  --local                Apply configuration locally.
  --include-untracked    Include untracked files in stash.
"""
    
    print("    Subcommand categories:")
    subcommand_categories = {
        'Repository': ['init', 'clone'],
        'Working Directory': ['add', 'commit', 'status'],
        'Branching': ['branch', 'checkout', 'merge'],
        'Remote': ['push', 'pull', 'remote'],
        'History': ['log'],
        'Configuration': ['config'],
        'Stashing': ['stash']
    }
    
    for category, commands in subcommand_categories.items():
        print(f"      {category}: {', '.join(commands)}")
    
    # 2. Test subcommand parsing
    print("  2. Subcommand parsing tests:")
    
    subcommand_tests = [
        # Repository operations
        (['init'], 'Initialize repository'),
        (['init', '/my/repo', '--bare'], 'Initialize bare repository'),
        (['clone', 'https://github.com/user/repo.git', 'local-dir'], 'Clone repository'),
        (['clone', 'git@github.com:user/repo.git', '--depth=1', '--branch=develop'], 'Shallow clone'),
        
        # Working directory operations  
        (['add', 'file1.py', 'file2.js'], 'Add specific files'),
        (['add', '*.py', '--all', '--dry-run'], 'Add with options'),
        (['commit', '-m', 'Fix bug in parser'], 'Simple commit'),
        (['commit', 'file1.py', 'file2.py', '--message=Update files', '--amend'], 'Commit specific files'),
        
        # Branching operations
        (['branch', 'feature-x'], 'Create branch'),
        (['branch', '--delete', 'old-feature'], 'Delete branch'),
        (['branch', '--rename', 'old-name', 'new-name'], 'Rename branch'),
        (['checkout', 'main', '--force'], 'Force checkout'),
        (['merge', 'feature-branch', '--no-ff'], 'Merge with options'),
        
        # Remote operations
        (['push', 'origin', 'main', '--force'], 'Force push'),
        (['pull', '--rebase'], 'Pull with rebase'),
        (['remote', 'add', 'upstream', 'https://github.com/original/repo.git'], 'Add remote'),
        (['remote', 'list', '--verbose'], 'List remotes'),
        
        # Other operations
        (['status', '--short', '--show-branch'], 'Short status'),
        (['log', '--oneline', '--graph', '-n', '5'], 'Graph log'),
        (['config', 'set', 'user.name', 'John Doe', '--global'], 'Set global config'),
        (['stash', 'push', '--include-untracked'], 'Stash with untracked'),
    ]
    
    for i, (args, description) in enumerate(subcommand_tests):
        try:
            print(f"    Test {i+1}: {description}")
            print(f"      Command: vcs {' '.join(args)}")
            
            parsed = docopt(git_like_doc, argv=args)
            
            # Identify the main command and subcommands
            main_commands = []
            sub_commands = []
            arguments = []
            options = []
            
            for key, value in parsed.items():
                if value:
                    if key.islower() and not key.startswith('<') and not key.startswith('-'):
                        if key in ['init', 'clone', 'add', 'commit', 'branch', 'checkout', 'merge', 'push', 'pull', 'status', 'log', 'remote', 'config', 'stash']:
                            main_commands.append(key)
                        else:
                            sub_commands.append(key)
                    elif key.startswith('<'):
                        arguments.append(f"{key}={value}")
                    elif key.startswith('-'):
                        if value == True:
                            options.append(key)
                        else:
                            options.append(f"{key}={value}")
            
            print(f"        Main command: {main_commands[0] if main_commands else 'none'}")
            assert parsed[args[0]] is True and args[0] in main_commands, (args, main_commands)
            if sub_commands:
                print(f"        Subcommands: {sub_commands}")
            if arguments:
                print(f"        Arguments: {arguments}")
            if options:
                print(f"        Options: {options[:3]}{'...' if len(options) > 3 else ''}")
                
        except SystemExit:
            print(f"        System exit (validation error)")
        except Exception as e:
            print(f"        Parse error: {type(e).__name__}")
            raise
        
        print()
    
    # 3. Command hierarchy analysis
    print("  3. Command hierarchy analysis:")
    
    # Analyze the hierarchical structure
    try:
        # Test different command structures to understand hierarchy
        hierarchy_tests = [
            (['remote', 'add', 'origin', 'url'], 'remote -> add'),
            (['config', 'set', 'key', 'value'], 'config -> set'),
            (['config', 'get', 'key'], 'config -> get'),
            (['stash', 'push'], 'stash -> push'),
        ]
        
        for args, structure in hierarchy_tests:
            try:
                parsed = docopt(git_like_doc, argv=args)
                
                print(f"    Structure: {structure}")
                print(f"      Command: vcs {' '.join(args)}")
                
                # Find which parts are True (indicating they were matched)
                matched_parts = []
                for key, value in parsed.items():
                    if value == True and key.islower() and not key.startswith('<') and not key.startswith('-'):
                        matched_parts.append(key)
                
                print(f"      Matched parts: {' -> '.join(matched_parts) if matched_parts else 'none'}")
                assert set(structure.split(' -> ')) <= set(matched_parts), (structure, matched_parts)
                
                # Show arguments for this command structure
                args_found = [(k, v) for k, v in parsed.items() if v and k.startswith('<')]
                if args_found:
                    print(f"      Arguments: {[f'{k}={v}' for k, v in args_found]}")
                
            except Exception as e:
                print(f"      Analysis failed: {type(e).__name__}")
                raise
            
            print()
    
    except Exception as e:
        print(f"    Hierarchy analysis failed: {type(e).__name__}")
        raise  # the inner handler re-raises; swallowing it here would undo that
    
    # 4. Complex option interactions
    print("  4. Complex option interactions:")
    
    # Test how different options interact with each other
    interaction_tests = [
        (['add', 'file.py', '--all', '--force', '--dry-run'], 'Multiple add flags'),
        (['commit', '--message=Fix', '--all', '--amend'], 'Multiple commit options'),
        (['log', '--oneline', '--graph', '--since=2023-01-01', '-n', '20'], 'Multiple log options'),
        (['push', 'origin', 'main', '--force', '--tags', '--dry-run'], 'Multiple push options'),
    ]
    
    for args, description in interaction_tests:
        try:
            print(f"    Test: {description}")
            parsed = docopt(git_like_doc, argv=args)
            
            # Count different types of options
            flags = [k for k, v in parsed.items() if v == True and k.startswith('-')]
            valued_options = [k for k, v in parsed.items() if v and v != True and k.startswith('-')]
            
            print(f"      Boolean flags: {len(flags)} ({flags[:3]}{'...' if len(flags) > 3 else ''})")
            print(f"      Valued options: {len(valued_options)} ({[f'{k}={parsed[k]}' for k in valued_options[:2]]}{'...' if len(valued_options) > 2 else ''})")
            # every option on the command line is set in the result (flags True, valued options their value)
            assert all(parsed[a.split('=')[0]] for a in args if a.startswith('-')), (args, flags, valued_options)
            
        except Exception as e:
            print(f"      Interaction test failed: {type(e).__name__}")
            raise
        
        print()
    
    # 5. Docstring structure validation
    print("  5. Docstring structure validation:")
    
    # Validate the structure of our complex docstring
    doc_sections = {}
    current_section = None
    
    for line in git_like_doc.split('\n'):
        line = line.strip()
        if line.endswith(':') and line[:-1].isalpha():
            current_section = line[:-1]
            doc_sections[current_section] = []
        elif line and current_section:
            doc_sections[current_section].append(line)
    
    print(f"    Docstring sections found:")
    for section, content in doc_sections.items():
        print(f"      {section}: {len(content)} lines")
    
    # Count usage patterns
    if 'Usage' in doc_sections:
        usage_patterns = [line for line in doc_sections['Usage'] if line.startswith('vcs ')]
        print(f"    Usage patterns: {len(usage_patterns)}")
        
        # Analyze pattern complexity
        pattern_complexity = {
            'simple': 0,  # Just command + args
            'moderate': 0,  # Command + few options
            'complex': 0  # Command + many options/subcommands
        }
        
        for pattern in usage_patterns:
            option_count = pattern.count('[') + pattern.count('<')
            if option_count <= 2:
                pattern_complexity['simple'] += 1
            elif option_count <= 5:
                pattern_complexity['moderate'] += 1
            else:
                pattern_complexity['complex'] += 1
        
        print(f"    Pattern complexity:")
        for complexity, count in pattern_complexity.items():
            print(f"      {complexity}: {count}")
    
    # Count options
    if 'Options' in doc_sections:
        option_lines = [line for line in doc_sections['Options'] if line.startswith(('-', ' -'))]
        print(f"    Total options defined: {len(option_lines)}")
        
        short_options = len([line for line in option_lines if ' -' in line and not line.startswith('  --')])
        long_options = len([line for line in option_lines if '--' in line])
        
        print(f"      Short options: {short_options}")
        print(f"      Long options: {long_options}")
    
    print("✓ Subcommands and complex structures completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"docopt example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Real-World Application Integration and Patterns

In [ ]:
from docopt import docopt
import sys
import json
from pathlib import Path

print("Example 4: Real-World Application Integration and Patterns")

try:
    print("✓ Real-world application patterns:")
    
    # 1. Complete application with docopt integration
    print("  1. Complete CLI application structure:")
    
    # Define a realistic application docstring
    app_doc = """DataProcessor - Advanced Data Processing CLI Tool

Usage:
  dataproc input validate <file> [--schema=<file>] [--format=<fmt>] [--strict]
  dataproc input convert <source> <target> [--format=<fmt>] [--batch-size=<n>] [--parallel=<workers>]
  dataproc transform apply <file> <operations>... [--output=<file>] [--preview] [--backup]
  dataproc transform batch <directory> <operations>... [--pattern=<glob>] [--recursive] [--output-dir=<dir>]
  dataproc analyze summary <file> [--columns=<cols>...] [--output=<file>] [--format=<fmt>]
  dataproc analyze correlation <file> <column1> <column2> [--method=<method>] [--threshold=<value>]
  dataproc export generate <file> <format> [--template=<file>] [--config=<file>] [--destination=<path>]
  dataproc export schedule <file> <format> <schedule> [--email=<addr>] [--webhook=<url>]
  dataproc config show [--section=<name>] [--format=<fmt>]
  dataproc config edit [--editor=<cmd>]
  dataproc config reset [--confirm]
  dataproc server start [--host=<addr>] [--port=<num>] [--workers=<n>] [--daemon]
  dataproc server stop [--force] [--timeout=<sec>]
  dataproc server status [--detailed] [--json]
  dataproc (-h | --help)
  dataproc --version

Arguments:
  <file>           Input/output file path
  <source>         Source file or directory
  <target>         Target file or directory
  <directory>      Directory to process
  <operations>     Data transformation operations (filter, map, reduce, etc.)
  <column1>        First column for correlation analysis
  <column2>        Second column for correlation analysis
  <format>         Output format (json, csv, xml, parquet)
  <schedule>       Cron-style schedule expression
  <addr>           Network address
  <num>            Port number
  <sec>            Timeout in seconds

Options:
  -h --help                    Show this help screen
  --version                    Show version information
  --schema=<file>              JSON schema file for validation
  --format=<fmt>               Data format (auto, json, csv, xml, parquet) [default: auto]
  --strict                     Enable strict validation mode
  --batch-size=<n>             Batch processing size [default: 1000]
  --parallel=<workers>         Number of parallel workers [default: 4]
  --output=<file>              Output file path
  --preview                    Show preview without executing
  --backup                     Create backup before processing
  --pattern=<glob>             File pattern for batch processing [default: *]
  --recursive                  Process directories recursively
  --output-dir=<dir>           Output directory for batch processing
  --columns=<cols>             Specific columns to analyze (comma-separated)
  --method=<method>            Correlation method (pearson, spearman, kendall) [default: pearson]
  --threshold=<value>          Correlation threshold [default: 0.5]
  --template=<file>            Template file for export
  --config=<file>              Configuration file
  --destination=<path>         Export destination path
  --email=<addr>               Email address for scheduled exports
  --webhook=<url>              Webhook URL for notifications
  --section=<name>             Configuration section name
  --editor=<cmd>               Text editor command [default: nano]
  --confirm                    Confirm destructive operations
  --host=<addr>                Server host address [default: localhost]
  --port=<num>                 Server port number [default: 8080]
  --workers=<n>                Number of server workers [default: 1]
  --daemon                     Run server as daemon
  --force                      Force operation without confirmation
  --timeout=<sec>              Operation timeout [default: 30]
  --detailed                   Show detailed information
  --json                       Output in JSON format

Examples:
  dataproc input validate data.csv --schema=schema.json --strict
  dataproc transform apply data.json filter,map,reduce --output=result.json --preview
  dataproc analyze summary dataset.csv --columns=age,income,education --format=json
  dataproc server start --host=0.0.0.0 --port=8080 --workers=4 --daemon
"""
    
    print("    Application structure:")
    app_modules = {
        'input': ['validate', 'convert'],
        'transform': ['apply', 'batch'],
        'analyze': ['summary', 'correlation'],
        'export': ['generate', 'schedule'],
        'config': ['show', 'edit', 'reset'],
        'server': ['start', 'stop', 'status']
    }
    
    for module, commands in app_modules.items():
        print(f"      {module}: {', '.join(commands)}")
    
    # 2. Application command dispatcher
    print("  2. Command dispatcher pattern:")
    
    class DataProcessorCLI:
        def __init__(self):
            self.config = {'batch_size': 1000, 'workers': 4, 'host': 'localhost'}
            
        def dispatch_command(self, args):
            """Dispatch commands based on parsed arguments"""
            try:
                parsed = docopt(app_doc, argv=args)
                
                # Identify main module and command
                modules = ['input', 'transform', 'analyze', 'export', 'config', 'server']
                active_module = None
                active_command = None
                
                for module in modules:
                    if parsed.get(module):
                        active_module = module
                        break
                
                if active_module:
                    # Find the specific command within the module
                    module_commands = app_modules.get(active_module, [])
                    for command in module_commands:
                        if parsed.get(command):
                            active_command = command
                            break
                
                return {
                    'module': active_module,
                    'command': active_command,
                    'args': parsed,
                    'success': True
                }
                
            except SystemExit as e:
                return {'success': False, 'error': 'SystemExit', 'code': e.code if hasattr(e, 'code') else 1}
            except Exception as e:
                return {'success': False, 'error': str(e), 'type': type(e).__name__}
        
        def execute_command(self, dispatch_result):
            """Execute the dispatched command"""
            if not dispatch_result['success']:
                return f"Command failed: {dispatch_result['error']}"
            
            module = dispatch_result['module']
            command = dispatch_result['command']
            args = dispatch_result['args']
            
            # Simulate command execution
            if module == 'input' and command == 'validate':
                file_path = args['<file>']
                schema = args.get('--schema', 'default')
                strict = args.get('--strict', False)
                return f"Validating {file_path} with schema {schema} (strict: {strict})"
            
            elif module == 'transform' and command == 'apply':
                file_path = args['<file>']
                operations = args['<operations>']
                preview = args.get('--preview', False)
                return f"Applying {len(operations)} operations to {file_path} (preview: {preview})"
            
            elif module == 'analyze' and command == 'summary':
                file_path = args['<file>']
                columns = args.get('--columns', [])
                return f"Analyzing {file_path} summary for {len(columns) if columns else 'all'} columns"
            
            elif module == 'server' and command == 'start':
                host = args.get('--host', 'localhost')
                port = args.get('--port', '8080')
                workers = args.get('--workers', '1')
                return f"Starting server on {host}:{port} with {workers} workers"
            
            else:
                return f"Executing {module}.{command} with {len([k for k, v in args.items() if v])} parameters"
    
    # Test the dispatcher
    cli = DataProcessorCLI()
    
    test_commands = [
        ['input', 'validate', 'data.csv', '--schema=schema.json', '--strict'],
        ['transform', 'apply', 'dataset.json', 'filter', 'map', '--preview'],
        ['analyze', 'summary', 'stats.csv', '--columns=age,income', '--format=json'],
        ['server', 'start', '--host=0.0.0.0', '--port=9000', '--workers=8'],
        ['config', 'show', '--section=database'],
    ]
    
    for i, cmd_args in enumerate(test_commands):
        print(f"    Command {i+1}: {' '.join(cmd_args)}")
        
        dispatch_result = cli.dispatch_command(cmd_args)
        # every test command matches a usage pattern, so it must dispatch to its own module and command
        assert dispatch_result['success'], f"dispatch of {cmd_args!r} failed: {dispatch_result}"
        assert (dispatch_result['module'], dispatch_result['command']) == (cmd_args[0], cmd_args[1]), dispatch_result
        print(f"      Module: {dispatch_result['module']}")
        print(f"      Command: {dispatch_result['command']}")
        
        execution_result = cli.execute_command(dispatch_result)
        print(f"      Result: {execution_result}")
        
        print()
    
    # 3. Configuration management integration
    print("  3. Configuration management:")
    
    # Test configuration-related commands
    config_tests = [
        (['config', 'show'], 'Show all configuration'),
        (['config', 'show', '--section=server', '--format=json'], 'Show server config as JSON'),
        (['config', 'edit', '--editor=vim'], 'Edit config with vim'),
        (['config', 'reset', '--confirm'], 'Reset configuration'),
    ]
    
    for cmd_args, description in config_tests:
        try:
            print(f"    Config test: {description}")
            parsed = docopt(app_doc, argv=cmd_args)
            
            # Extract configuration-specific options
            config_options = {}
            if parsed.get('--section'):
                config_options['section'] = parsed['--section']
            if parsed.get('--format'):
                config_options['format'] = parsed['--format']
            if parsed.get('--editor'):
                config_options['editor'] = parsed['--editor']
            if parsed.get('--confirm'):
                config_options['confirm'] = True
            
            print(f"      Command: config {parsed.get('show', '') or parsed.get('edit', '') or parsed.get('reset', '')}")
            print(f"      Options: {config_options if config_options else 'none'}")
            
        except Exception as e:
            print(f"      Config test failed: {type(e).__name__}")
            raise
        
        print()
    
    # 4. Error handling patterns
    print("  4. Error handling patterns:")
    
    def safe_docopt_parse(doc, args):
        """Safe docopt parsing with error handling"""
        try:
            result = docopt(doc, argv=args)
            return {'success': True, 'parsed': result, 'error': None}
        except SystemExit as e:
            return {'success': False, 'parsed': None, 'error': 'SystemExit', 'code': getattr(e, 'code', 1)}
        except Exception as e:
            return {'success': False, 'parsed': None, 'error': str(e), 'type': type(e).__name__}
    
    # Test error scenarios: each argv matches no usage pattern, so docopt must reject it (DocoptExit is a SystemExit)
    error_scenarios = [
        ([], 'No arguments provided'),
        (['invalid'], 'Invalid command'),
        (['input'], 'Missing subcommand'),
        (['input', 'validate'], 'Missing required argument'),
        (['transform', 'apply', 'file.json', '--invalid-option'], 'Invalid option'),
    ]
    
    for args, scenario in error_scenarios:
        print(f"    Error scenario: {scenario}")
        print(f"      Command: {' '.join(args) if args else '(empty)'}")
        
        result = safe_docopt_parse(app_doc, args)
        assert not result['success'] and result['error'] == 'SystemExit', f"docopt accepted argv {args!r}: {result}"
        print(f"      Expected failure: {result['error']}")
        
        print()
    
    # docopt does not type-check option VALUES (measured: docopt 0.6.2 parses `server start --port=invalid`
    # with --port == 'invalid'), so an invalid value is the application's to reject, not docopt's
    result = safe_docopt_parse(app_doc, ['server', 'start', '--port=invalid'])
    assert result['success'] and result['parsed']['--port'] == 'invalid', result
    print("    Option value: docopt passes --port='invalid' through unchanged; the application must validate it")
    
    # 5. Performance and scalability patterns
    print("  5. Performance patterns:")
    
    # Test batch processing configurations with safer commands
    batch_tests = [
        (['input', 'convert', 'large.csv', 'output.json', '--batch-size=10000', '--parallel=16'], 'Large file conversion'),
        (['server', 'start', '--workers=32', '--host=0.0.0.0'], 'High-concurrency server'),
        (['analyze', 'summary', 'dataset.csv', '--format=json'], 'Analysis with JSON output'),
    ]
    
    for args, description in batch_tests:
        try:
            print(f"    Performance test: {description}")
            parsed = docopt(app_doc, argv=args)
            
            # Extract performance-related settings
            perf_settings = {}
            if parsed.get('--batch-size'):
                perf_settings['batch_size'] = int(parsed['--batch-size'])
            if parsed.get('--parallel'):
                perf_settings['parallel_workers'] = int(parsed['--parallel'])
            if parsed.get('--workers'):
                perf_settings['server_workers'] = int(parsed['--workers'])
            
            print(f"      Performance settings: {perf_settings}")
            
            # Calculate estimated throughput (simulated)
            if 'batch_size' in perf_settings and 'parallel_workers' in perf_settings:
                estimated_throughput = perf_settings['batch_size'] * perf_settings['parallel_workers']
                print(f"      Estimated throughput: {estimated_throughput:,} records/batch")
            
        except Exception as e:
            print(f"      Performance test failed: {type(e).__name__}")
            raise
        
        print()
    
    # 6. Integration summary
    print("  6. Integration summary:")
    
    # Analyze the full application structure
    total_commands = 0
    total_options = 0
    
    # Count commands from docstring
    usage_lines = [line.strip() for line in app_doc.split('\n') 
                   if line.strip().startswith('dataproc ') and not line.strip().startswith('dataproc (-h')]
    total_commands = len(usage_lines)
    
    # Count options from docstring
    option_lines = [line.strip() for line in app_doc.split('\n') 
                   if line.strip().startswith('--') and ('=' in line or 'Show' in line)]
    total_options = len(option_lines)
    
    print(f"    Application complexity:")
    print(f"      Total modules: {len(app_modules)}")
    print(f"      Total commands: {total_commands}")
    print(f"      Total options: {total_options}")
    print(f"      Docstring length: {len(app_doc)} characters")
    
    print(f"    Integration benefits:")
    benefits = [
        "✓ Single source of truth for CLI interface",
        "✓ Automatic help generation from docstring",
        "✓ Consistent argument validation",
        "✓ Easy command dispatching",
        "✓ Self-documenting code structure",
        "✓ Unix philosophy compliance"
    ]
    
    for benefit in benefits:
        print(f"      {benefit}")
    
    print("✓ Real-world application integration patterns completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"docopt example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Advanced Features and Best Practices

In [ ]:
from docopt import docopt
import re
import os

print("Example 5: Advanced Features and Best Practices")

try:
    print("✓ Advanced features and best practices:")
    
    # 1. Docstring validation and best practices
    print("  1. Docstring validation and best practices:")
    
    # Example of well-structured docstring following docopt best practices
    best_practice_doc = """CloudSync - Professional Cloud Synchronization Tool

Usage:
  cloudsync auth login [--provider=<name>] [--interactive] [--config=<file>]
  cloudsync auth logout [--all] [--preserve-config]
  cloudsync auth status [--verbose] [--json]
  cloudsync sync upload <local-path> <remote-path> [--recursive] [--exclude=<patterns>...] [--dry-run] [--progress]
  cloudsync sync download <remote-path> <local-path> [--recursive] [--include=<patterns>...] [--resume] [--verify]
  cloudsync sync bidirectional <path> [--conflict-resolution=<strategy>] [--watch] [--interval=<seconds>]
  cloudsync list [<path>] [--recursive] [--show-hidden] [--sort=<field>] [--format=<type>]
  cloudsync info <path> [--metadata] [--show-permissions] [--versions]
  cloudsync share create <path> [--permissions=<perms>] [--expiry=<date>] [--password=<pass>]
  cloudsync share revoke <share-id> [--force]
  cloudsync share list [--active-only] [--format=<type>]
  cloudsync config get <key> [--section=<name>] [--default=<value>]
  cloudsync config set <key> <value> [--section=<name>] [--type=<datatype>]
  cloudsync config list [--section=<name>] [--show-defaults]
  cloudsync monitor start [--background] [--log-level=<level>]
  cloudsync monitor stop [--graceful] [--timeout=<seconds>]
  cloudsync monitor status [--detailed]
  cloudsync (-h | --help)
  cloudsync --version

Arguments:
  <local-path>           Local file or directory path
  <remote-path>          Remote file or directory path  
  <path>                 Generic path (local or remote)
  <name>                 Provider name (aws, gcp, azure, dropbox)
  <file>                 Configuration file path
  <patterns>             File patterns for filtering (glob syntax)
  <strategy>             Conflict resolution strategy (newest, oldest, manual, skip)
  <seconds>              Time interval in seconds
  <field>                Sort field (name, size, modified, type)
  <type>                 Output format (table, json, csv, xml)
  <perms>                Permission level (read, write, admin)
  <date>                 Expiry date (YYYY-MM-DD format)
  <pass>                 Password for shared link
  <share-id>             Unique identifier for shared resource
  <key>                  Configuration key name
  <value>                Configuration value
  <datatype>             Value data type (string, int, bool, float)
  <level>                Log level (debug, info, warn, error)

Options:
  -h --help                           Show this help message and exit
  --version                           Show version information and exit
  --provider=<name>                   Cloud provider name [default: auto]
  --interactive                       Enable interactive authentication mode
  --config=<file>                     Configuration file path [default: ~/.cloudsync/config.json]
  --all                              Apply to all providers/sessions
  --preserve-config                   Keep configuration after logout
  --verbose                          Enable verbose output
  --json                             Output in JSON format
  --recursive                        Process directories recursively
  --exclude=<patterns>               Exclude files matching patterns (repeatable)
  --include=<patterns>               Include only files matching patterns (repeatable)
  --dry-run                          Show what would be done without executing
  --progress                         Show progress indicator
  --resume                           Resume interrupted transfers
  --verify                           Verify file integrity after transfer
  --conflict-resolution=<strategy>   How to handle conflicts [default: newest]
  --watch                            Continuously monitor for changes
  --interval=<seconds>               Monitoring interval [default: 60]
  --show-hidden                      Include hidden files and directories
  --sort=<field>                     Sort output by field [default: name]
  --format=<type>                    Output format [default: table]
  --metadata                         Include file metadata
  --show-permissions                 Show file permissions
  --versions                         Show file version history
  --permissions=<perms>              Set permissions level [default: read]
  --expiry=<date>                    Share expiry date
  --password=<pass>                  Password protect shared link
  --force                            Force operation without confirmation
  --active-only                      Show only active shares
  --section=<name>                   Configuration section [default: general]
  --default=<value>                  Default value if key not found
  --type=<datatype>                  Data type for configuration value [default: string]
  --show-defaults                    Include default values in output
  --background                       Run in background mode
  --log-level=<level>                Set logging level [default: info]
  --graceful                         Graceful shutdown
  --timeout=<seconds>                Operation timeout [default: 30]
  --detailed                         Show detailed status information

Environment Variables:
  CLOUDSYNC_CONFIG_PATH             Override default config file path
  CLOUDSYNC_PROVIDER                Default cloud provider
  CLOUDSYNC_LOG_LEVEL              Default logging level
  CLOUDSYNC_CACHE_DIR              Cache directory path

Examples:
  cloudsync auth login --provider=aws --interactive
  cloudsync sync upload ./documents /backup/docs --recursive --progress
  cloudsync sync download /shared/photos ./photos --include="*.jpg" --include="*.png"
  cloudsync sync bidirectional ~/CloudFolder --conflict-resolution=manual --watch
  cloudsync list /projects --recursive --sort=modified --format=json
  cloudsync share create /important/report.pdf --permissions=read --expiry=2024-12-31
  cloudsync config set sync.chunk_size 8192 --type=int --section=performance
  cloudsync monitor start --background --log-level=debug

Exit Codes:
  0    Success
  1    General error
  2    Authentication required
  3    Network error
  4    File not found
  5    Permission denied
  6    Configuration error
  7    Service unavailable
"""
    
    print("    Docstring quality analysis:")
    
    # Analyze docstring structure
    sections = {}
    current_section = None
    
    for line in best_practice_doc.split('\n'):
        line = line.strip()
        if line.endswith(':') and line.replace(':', '').replace(' ', '').isalpha():
            section_name = line[:-1]
            current_section = section_name
            sections[section_name] = []
        elif line and current_section:
            sections[current_section].append(line)
    
    print(f"      Documentation sections: {list(sections.keys())}")
    
    for section, content in sections.items():
        if section in ['Usage', 'Arguments', 'Options']:
            print(f"        {section}: {len(content)} items")
        else:
            print(f"        {section}: {len(content)} lines")
    
    # Validate usage patterns
    if 'Usage' in sections:
        usage_patterns = [line for line in sections['Usage'] if line.startswith('cloudsync ')]
        
        # Check for pattern consistency
        pattern_analysis = {
            'total_patterns': len(usage_patterns),
            'with_subcommands': 0,
            'with_options': 0,
            'with_required_args': 0,
            'with_optional_args': 0,
        }
        
        for pattern in usage_patterns:
            if ' ' in pattern.split('cloudsync ')[1] and not pattern.split('cloudsync ')[1].startswith('(-h'):
                pattern_analysis['with_subcommands'] += 1
            if '[--' in pattern or ' --' in pattern:
                pattern_analysis['with_options'] += 1
            if '<' in pattern and '>' in pattern:
                pattern_analysis['with_required_args'] += 1
            if '[' in pattern and ']' in pattern:
                pattern_analysis['with_optional_args'] += 1
        
        print(f"      Usage pattern analysis:")
        for pattern_type, count in pattern_analysis.items():
            percentage = (count / pattern_analysis['total_patterns'] * 100) if pattern_analysis['total_patterns'] > 0 else 0
            print(f"        {pattern_type.replace('_', ' ').title()}: {count} ({percentage:.1f}%)")
    
    # 2. Advanced pattern matching and validation
    print("  2. Advanced pattern matching:")
    
    # Test complex pattern matching scenarios
    complex_patterns = [
        # Authentication commands
        (['auth', 'login', '--provider=aws', '--interactive'], 'Interactive AWS login'),
        (['auth', 'status', '--verbose', '--json'], 'Detailed auth status'),
        
        # Sync operations with multiple options
        (['sync', 'upload', './src', '/backup/src', '--recursive', '--exclude=*.pyc', '--exclude=__pycache__', '--progress'], 'Upload with exclusions'),
        (['sync', 'bidirectional', '~/Documents', '--conflict-resolution=manual', '--watch', '--interval=30'], 'Bidirectional sync with monitoring'),
        
        # Complex listing and info
        (['list', '/projects', '--recursive', '--show-hidden', '--sort=modified', '--format=json'], 'Advanced directory listing'),
        (['info', '/data/report.xlsx', '--metadata', '--show-permissions', '--versions'], 'Comprehensive file info'),
        
        # Share management
        (['share', 'create', '/public/document.pdf', '--permissions=read', '--expiry=2024-12-31', '--password=secret123'], 'Protected share creation'),
        
        # Configuration with type specification
        (['config', 'set', 'performance.chunk_size', '16384', '--type=int', '--section=advanced'], 'Typed configuration'),
        
        # Monitor operations
        (['monitor', 'start', '--background', '--log-level=debug'], 'Background monitoring'),
    ]
    
    for i, (args, description) in enumerate(complex_patterns):
        try:
            print(f"    Pattern {i+1}: {description}")
            print(f"      Command: cloudsync {' '.join(args)}")
            
            parsed = docopt(best_practice_doc, argv=args)
            
            # Analyze the parsed result complexity
            active_elements = [(k, v) for k, v in parsed.items() if v and v != 'auto' and v != 'general']
            
            # Categorize elements
            commands = [k for k, v in active_elements if k.islower() and not k.startswith('<') and not k.startswith('-')]
            arguments = [k for k, v in active_elements if k.startswith('<')]
            options = [k for k, v in active_elements if k.startswith('-')]
            
            print(f"        Active elements: {len(active_elements)}")
            print(f"        Commands: {commands}")
            if arguments:
                print(f"        Arguments: {[f'{k}={parsed[k]}' for k in arguments[:2]]}{'...' if len(arguments) > 2 else ''}")
            if options:
                option_summary = []
                for k in options[:3]:
                    val = parsed[k]
                    if val == True:
                        option_summary.append(k)
                    else:
                        option_summary.append(f'{k}={val}')
                print(f"        Options: {option_summary}{'...' if len(options) > 3 else ''}")
            
        except Exception as e:
            print(f"        Pattern matching failed: {type(e).__name__}")
            raise
        
        print()
    
    # 3. Error handling and validation best practices
    print("  3. Error handling best practices:")
    
    def validate_docopt_args(parsed_args, validation_rules):
        """Validate parsed arguments against business rules"""
        errors = []
        warnings = []
        
        # Example validation rules
        if validation_rules.get('require_auth_for_sync', False):
            if any(parsed_args.get(k) for k in ['upload', 'download', 'bidirectional']):
                # In real app, check if user is authenticated
                warnings.append("Sync operations require authentication")
        
        if parsed_args.get('--interval'):
            try:
                interval = int(parsed_args['--interval'])
                if interval < 10:
                    errors.append("Monitoring interval must be at least 10 seconds")
                elif interval > 3600:
                    warnings.append("Very long monitoring intervals may not be practical")
            except ValueError:
                errors.append("Monitoring interval must be a valid number")
        
        if parsed_args.get('--expiry'):
            expiry = parsed_args['--expiry']
            if not re.match(r'^\d{4}-\d{2}-\d{2}$', expiry):
                errors.append("Expiry date must be in YYYY-MM-DD format")
        
        if parsed_args.get('--timeout'):
            try:
                timeout = int(parsed_args['--timeout'])
                if timeout < 1:
                    errors.append("Timeout must be at least 1 second")
                elif timeout > 300:
                    warnings.append("Very long timeouts may not be practical")
            except ValueError:
                errors.append("Timeout must be a valid number")
        
        return {'errors': errors, 'warnings': warnings}
    
    # Test validation on parsed commands
    validation_tests = [
        (['sync', 'upload', 'file.txt', '/backup/file.txt'], 'Sync without auth check'),
        (['sync', 'bidirectional', '/folder', '--interval=5'], 'Invalid monitoring interval'),
        (['sync', 'bidirectional', '/folder', '--interval=7200'], 'Very long interval'),
        (['monitor', 'stop', '--timeout=0'], 'Invalid timeout value'),
        (['share', 'create', '/file.pdf', '--expiry=2024/12/31'], 'Invalid date format'),
        (['share', 'create', '/file.pdf', '--expiry=2024-12-31'], 'Valid date format'),
    ]
    
    validation_rules = {'require_auth_for_sync': True}
    
    for args, test_description in validation_tests:
        try:
            print(f"    Validation test: {test_description}")
            parsed = docopt(best_practice_doc, argv=args)
            
            validation_result = validate_docopt_args(parsed, validation_rules)
            
            if validation_result['errors']:
                print(f"      Errors: {validation_result['errors']}")
            if validation_result['warnings']:
                print(f"      Warnings: {validation_result['warnings']}")
            if not validation_result['errors'] and not validation_result['warnings']:
                print(f"      Validation: Passed")
            
        except Exception as e:
            print(f"      Parse error: {type(e).__name__}")
            raise
        
        print()
    
    # 4. Performance considerations and optimization
    print("  4. Performance considerations:")
    
    # Measure parsing performance for complex docstrings
    import time
    
    performance_tests = [
        (best_practice_doc, ['sync', 'upload', 'file.txt', '/remote/file.txt']),
        (best_practice_doc, ['config', 'set', 'key', 'value', '--section=test']),
        (best_practice_doc, ['list', '/path', '--recursive', '--format=json']),
    ]
    
    parse_times = []
    
    for doc, args in performance_tests:
        start_time = time.time()
        
        # Parse multiple times to get average
        iterations = 100
        for _ in range(iterations):
            try:
                docopt(doc, argv=args)
            except SystemExit:
                pass  # Expected for help/version
        
        end_time = time.time()
        avg_time = (end_time - start_time) / iterations * 1000  # Convert to milliseconds
        parse_times.append(avg_time)
    
    print(f"    Parsing performance:")
    print(f"      Average parse time: {sum(parse_times) / len(parse_times):.3f}ms")
    print(f"      Min parse time: {min(parse_times):.3f}ms")
    print(f"      Max parse time: {max(parse_times):.3f}ms")
    print(f"      Docstring complexity: {len(best_practice_doc)} characters")
    print(f"      Performance rating: {'Excellent' if sum(parse_times) / len(parse_times) < 1.0 else 'Good' if sum(parse_times) / len(parse_times) < 5.0 else 'Acceptable'}")
    
    # 5. Best practices summary and recommendations
    print("  5. Best practices summary:")
    
    best_practices = {
        'Docstring Structure': [
            'Include clear Usage section with all command patterns',
            'Document all arguments with descriptions',
            'Provide comprehensive Options section with defaults',
            'Add Examples section for common use cases',
            'Include Environment Variables if applicable',
            'Document Exit Codes for better debugging'
        ],
        'Pattern Design': [
            'Use consistent command hierarchy (module subcommand)',
            'Group related options logically',
            'Provide sensible defaults for optional parameters',
            'Use repeatable options (...) when appropriate',
            'Follow Unix conventions for option naming'
        ],
        'Error Handling': [
            'Wrap docopt() calls in try-except blocks',
            'Validate arguments after parsing',
            'Provide helpful error messages',
            'Use appropriate exit codes',
            'Handle SystemExit for help/version gracefully'
        ],
        'Code Organization': [
            'Create command dispatcher functions',
            'Separate parsing from business logic',
            'Use type hints for better code clarity',
            'Implement comprehensive logging',
            'Write unit tests for argument parsing'
        ],
        'User Experience': [
            'Provide clear, concise help text',
            'Use consistent naming conventions',
            'Include usage examples in docstring',
            'Support both short and long option forms',
            'Follow principle of least surprise'
        ]
    }
    
    print(f"    docopt Best Practices Guide:")
    for category, practices in best_practices.items():
        print(f"      {category}:")
        for practice in practices:
            print(f"        • {practice}")
        print()
    
    # Final capability summary
    print("  6. docopt Capability Summary:")
    
    capabilities = [
        "✓ Automatic CLI parsing from docstrings",
        "✓ Support for complex command hierarchies",
        "✓ Flexible option patterns (flags, values, repeatable)",
        "✓ Built-in help and version handling",
        "✓ Argument validation and type checking",
        "✓ Unix-style command-line conventions",
        "✓ Minimal boilerplate code required",
        "✓ Self-documenting interface definitions",
        "✓ Error handling with meaningful messages",
        "✓ Integration with existing Python applications"
    ]
    
    for capability in capabilities:
        print(f"      {capability}")
    
    print(f"    Package ecosystem integration:")
    integrations = [
        "Works with argparse for hybrid solutions",
        "Compatible with Click for complex applications",  
        "Integrates with configuration management libraries",
        "Supports testing frameworks for CLI validation",
        "Works with logging libraries for better debugging"
    ]
    
    for integration in integrations:
        print(f"      • {integration}")
    
    print("✓ Advanced features and best practices completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"docopt example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")